# Hypothyroidism Cohort & Treatment Response — All of Us Data Availability

**Objective:** Identify individuals diagnosed with hypothyroidism who have thyroid-related lab results,
assess T4 (levothyroxine) treatment exposure, and evaluate data availability for determining
treatment response via longitudinal labs and symptom changes.

**OMOP CDM tables used:**
- `condition_occurrence` — diagnoses
- `measurement` — lab results
- `drug_exposure` — medication exposures
- `observation` — observations, symptoms
- `person` — demographics
- `concept` / `concept_ancestor` — OMOP vocabulary and hierarchy lookups

## 1. Setup

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

# All of Us workbench sets this env variable to the versioned CDR dataset
DATASET = os.environ["WORKSPACE_CDR"]

client = bigquery.Client()

def run_query(sql: str) -> pd.DataFrame:
    """Execute a BigQuery SQL string and return a DataFrame."""
    return client.query(sql).to_dataframe()

print(f"CDR dataset: {DATASET}")

## 2. Hypothyroidism Concept Hierarchy

The OMOP condition hierarchy is captured in `concept_ancestor`. We anchor on the root
SNOMED standard concepts for hypothyroidism and pull **all descendants**, so child conditions
like myxedema, cretinism, thyroid atrophy, and post-ablative hypothyroidism are automatically
included without manual enumeration.

We also retain explicit ICD-9/ICD-10 source codes as a supplementary safety net for records
where source-to-standard mapping may be incomplete.

In [ ]:
# ---------------------------------------------------------------------------
# 2a. Identify root SNOMED standard concept IDs for hypothyroidism.
#
# We use two roots:
#   - "Hypothyroidism" (SNOMED 40930001) — covers all hypothyroid subtypes
#   - "Hashimoto thyroiditis" (SNOMED 21983002) — autoimmune; may sit in a
#     separate branch (thyroiditis) rather than under hypothyroidism directly
#
# Running this cell confirms which concept IDs exist in this CDR version.
# ---------------------------------------------------------------------------
HYPO_ROOT_SNOMED_CODES = "'40930001', '21983002'"

sql_root_concepts = f"""
SELECT concept_id, concept_name, concept_code, standard_concept
FROM `{DATASET}.concept`
WHERE vocabulary_id = 'SNOMED'
  AND concept_code IN ({HYPO_ROOT_SNOMED_CODES})
"""

df_roots = run_query(sql_root_concepts)
print("Root concepts:")
display(df_roots)

# Build a comma-separated string of the integer concept IDs for use in SQL
ROOT_CONCEPT_IDS = ', '.join(str(i) for i in df_roots['concept_id'].tolist())
print(f"\nRoot concept IDs for concept_ancestor queries: {ROOT_CONCEPT_IDS}")

In [ ]:
# ---------------------------------------------------------------------------
# 2b. Enumerate the full hypothyroidism concept hierarchy via concept_ancestor.
#
# This is the complete set of SNOMED standard concepts that will be used to
# identify cohort members. Review this list to confirm expected child
# conditions are present (myxedema, cretinism, thyroid atrophy, etc.).
# ---------------------------------------------------------------------------
sql_all_hypo_concepts = f"""
SELECT
    c.concept_id,
    c.concept_name,
    c.concept_code       AS snomed_code,
    ca.min_levels_of_separation AS levels_from_root
FROM `{DATASET}.concept_ancestor` ca
JOIN `{DATASET}.concept` c ON ca.descendant_concept_id = c.concept_id
WHERE ca.ancestor_concept_id IN ({ROOT_CONCEPT_IDS})
  AND c.domain_id = 'Condition'
  AND c.standard_concept = 'S'
ORDER BY ca.min_levels_of_separation, c.concept_name
"""

df_all_hypo = run_query(sql_all_hypo_concepts)
print(f"Total hypothyroidism-family standard concepts (including descendants): {len(df_all_hypo)}")
print("\nSample of child conditions:")
df_all_hypo

In [ ]:
# ---------------------------------------------------------------------------
# 2c. ICD source codes — supplementary safety net.
#
# Used to catch condition_occurrence records whose condition_source_concept_id
# maps to an ICD code even when the standard concept mapping is absent or
# maps to a non-hypothyroid standard concept.
# ---------------------------------------------------------------------------
HYPO_ICD10 = (
    "'E00', 'E00.0', 'E00.1', 'E00.2', 'E00.9',"
    "'E01', 'E01.0', 'E01.1', 'E01.2', 'E01.8',"
    "'E02',"
    "'E03', 'E03.0', 'E03.1', 'E03.2', 'E03.3', 'E03.4', 'E03.5', 'E03.8', 'E03.9',"
    "'E06.3'"
)

HYPO_ICD9 = (
    "'243', '244', '244.0', '244.1', '244.2', '244.3', '244.8', '244.9'"
)

sql_src_concepts = f"""
SELECT concept_id, concept_name, vocabulary_id, concept_code
FROM `{DATASET}.concept`
WHERE (vocabulary_id = 'ICD10CM' AND concept_code IN ({HYPO_ICD10}))
   OR (vocabulary_id = 'ICD9CM'  AND concept_code IN ({HYPO_ICD9}))
ORDER BY vocabulary_id, concept_code
"""

df_src_concepts = run_query(sql_src_concepts)
print(f"ICD source concepts found: {len(df_src_concepts)}")
df_src_concepts

## 3. Reusable Cohort CTE

Define the cohort SQL fragment once as a Python string. Every subsequent query interpolates
`{COHORT_CTE}` to avoid duplication and keep the cohort definition consistent throughout.

**Matching logic (OR of two independent paths):**
1. **Standard concept path** — `condition_concept_id` is a descendant of a root hypothyroidism
   SNOMED concept via `concept_ancestor`. Captures the full condition hierarchy.
2. **Source code path** — `condition_source_concept_id` maps to a hypothyroidism ICD-9 or
   ICD-10 code. Safety net for records with incomplete standard-concept mapping.

In [ ]:
# Cohort CTE — injected into every downstream query via f-string.
# Returns: person_id, first_dx_date
COHORT_CTE = f"""
cohort AS (
    SELECT
        co.person_id,
        MIN(co.condition_start_date) AS first_dx_date
    FROM `{DATASET}.condition_occurrence` co
    WHERE
        -- Path 1: standard SNOMED concept is a descendant of a hypothyroidism root
        co.condition_concept_id IN (
            SELECT descendant_concept_id
            FROM `{DATASET}.concept_ancestor`
            WHERE ancestor_concept_id IN ({ROOT_CONCEPT_IDS})
        )
        OR
        -- Path 2: source concept is an ICD-10 or ICD-9 hypothyroidism code
        co.condition_source_concept_id IN (
            SELECT concept_id FROM `{DATASET}.concept`
            WHERE (vocabulary_id = 'ICD10CM' AND concept_code IN ({HYPO_ICD10}))
               OR (vocabulary_id = 'ICD9CM'  AND concept_code IN ({HYPO_ICD9}))
        )
    GROUP BY co.person_id
)
"""

# Levothyroxine (T4) CTE — reused in treatment and response queries
LEVO_FIRST_CTE = f"""
levo_first AS (
    SELECT de.person_id,
        MIN(de.drug_exposure_start_date) AS first_t4_date
    FROM `{DATASET}.drug_exposure` de
    WHERE de.drug_concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM `{DATASET}.concept` c
        JOIN `{DATASET}.concept_ancestor` ca ON c.concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id = 10582  -- levothyroxine RxNorm ingredient
          AND c.domain_id = 'Drug'
          AND c.standard_concept = 'S'
    )
    GROUP BY de.person_id
)
"""

# Liothyronine (T3) first prescription CTE — used for T3 non-responder classification
T3_FIRST_CTE = f"""
t3_first AS (
    SELECT de.person_id,
        MIN(de.drug_exposure_start_date) AS first_t3_date
    FROM `{DATASET}.drug_exposure` de
    WHERE de.drug_concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM `{DATASET}.concept` c
        JOIN `{DATASET}.concept_ancestor` ca ON c.concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id = 10597  -- liothyronine (T3) RxNorm ingredient
          AND c.domain_id = 'Drug'
          AND c.standard_concept = 'S'
    )
    GROUP BY de.person_id
)
"""

# TSH concept CTE
TSH_CONCEPTS_CTE = f"""
tsh_concepts AS (
    SELECT concept_id FROM `{DATASET}.concept`
    WHERE vocabulary_id = 'LOINC'
      AND concept_code IN ('3016-3', '11580-8')  -- TSH, TSH (sensitive)
)
"""

print("Cohort and reusable CTEs defined.")

## 4. Hypothyroidism Cohort

In [ ]:
sql_hypo_cohort = f"""
WITH {COHORT_CTE}
SELECT
    ch.person_id,
    ch.first_dx_date,
    p.birth_datetime,
    gc.concept_name AS gender,
    rc.concept_name AS race,
    ec.concept_name AS ethnicity
FROM cohort ch
JOIN `{DATASET}.person` p ON ch.person_id = p.person_id
LEFT JOIN `{DATASET}.concept` gc ON p.gender_concept_id    = gc.concept_id
LEFT JOIN `{DATASET}.concept` rc ON p.race_concept_id      = rc.concept_id
LEFT JOIN `{DATASET}.concept` ec ON p.ethnicity_concept_id = ec.concept_id
"""

df_cohort = run_query(sql_hypo_cohort)
print(f"Hypothyroidism cohort size: {len(df_cohort):,} individuals")
df_cohort.head()

In [ ]:
print("=== Gender distribution ===")
print(df_cohort['gender'].value_counts())
print("\n=== Race distribution ===")
print(df_cohort['race'].value_counts())

## 5. Thyroid Lab Concepts

In [ ]:
THYROID_LAB_LOINC = (
    "'3016-3',"   # TSH
    "'11580-8'"   # TSH (sensitive)
    ", '3024-7'"  # Free T4
    ", '3026-2'"  # Total T4
    ", '3051-0'"  # Free T3
    ", '3053-6'"  # Total T3
    ", '31161-1'" # Reverse T3
    ", '8099-1'"  # Anti-TPO
    ", '5380-5'"  # Anti-TPO (alternate)
    ", '22320-9'" # Anti-thyroglobulin
    ", '8098-3'"  # Anti-thyroglobulin (alternate)
    ", '14920-3'" # Thyroid stimulating immunoglobulin (TSI)
    ", '2986-8'"  # Thyroid binding globulin (TBG)
)

sql_lab_concepts = f"""
SELECT concept_id, concept_name, concept_code AS loinc_code
FROM `{DATASET}.concept`
WHERE vocabulary_id = 'LOINC'
  AND concept_code IN ({THYROID_LAB_LOINC})
ORDER BY concept_code
"""

df_lab_concepts = run_query(sql_lab_concepts)
print(f"Thyroid lab LOINC concepts found: {len(df_lab_concepts)}")
df_lab_concepts

## 6. Thyroid Lab Results for the Cohort

In [ ]:
sql_thyroid_labs = f"""
WITH {COHORT_CTE},
thyroid_lab_concepts AS (
    SELECT concept_id, concept_name, concept_code AS loinc_code
    FROM `{DATASET}.concept`
    WHERE vocabulary_id = 'LOINC'
      AND concept_code IN ({THYROID_LAB_LOINC})
)
SELECT
    m.person_id,
    m.measurement_date,
    tlc.concept_name  AS lab_name,
    tlc.loinc_code,
    m.value_as_number,
    m.range_low,
    m.range_high,
    uc.concept_name   AS unit
FROM `{DATASET}.measurement` m
JOIN cohort ch             ON m.person_id = ch.person_id
JOIN thyroid_lab_concepts tlc ON m.measurement_concept_id = tlc.concept_id
LEFT JOIN `{DATASET}.concept` uc ON m.unit_concept_id = uc.concept_id
ORDER BY m.person_id, m.measurement_date
"""

df_labs = run_query(sql_thyroid_labs)
print(f"Total thyroid lab records: {len(df_labs):,}")
print(f"Individuals with >= 1 thyroid lab: {df_labs['person_id'].nunique():,}")
df_labs.head()

In [ ]:
lab_summary = (
    df_labs.groupby('lab_name')
    .agg(
        records=('person_id', 'count'),
        unique_individuals=('person_id', 'nunique'),
        pct_with_value=('value_as_number', lambda x: (x.notna().sum() / len(x) * 100).round(1))
    )
    .sort_values('records', ascending=False)
)
print("=== Thyroid Lab Availability ===")
lab_summary

## 7. Levothyroxine Drug Concepts

In [ ]:
sql_drug_concepts = f"""
SELECT DISTINCT c.concept_id, c.concept_name, c.concept_class_id
FROM `{DATASET}.concept` c
JOIN `{DATASET}.concept_ancestor` ca ON c.concept_id = ca.descendant_concept_id
WHERE ca.ancestor_concept_id = 10582  -- levothyroxine (RxNorm ingredient)
  AND c.domain_id = 'Drug'
  AND c.standard_concept = 'S'
ORDER BY c.concept_class_id, c.concept_name
"""

df_drug_concepts = run_query(sql_drug_concepts)
print(f"Levothyroxine drug concepts (all formulations): {len(df_drug_concepts)}")
df_drug_concepts.head(20)

## 8. T4 Treatment Exposure

In [ ]:
sql_t4_treatment = f"""
WITH {COHORT_CTE},
{LEVO_FIRST_CTE}
SELECT
    ch.person_id,
    ch.first_dx_date,
    lf.first_t4_date,
    DATE_DIFF(lf.first_t4_date, ch.first_dx_date, DAY) AS days_dx_to_t4,
    COUNT(de.drug_exposure_id)       AS t4_exposure_records,
    MAX(de.drug_exposure_end_date)   AS last_t4_date,
    SUM(de.days_supply)              AS total_days_supply
FROM cohort ch
JOIN levo_first lf ON ch.person_id = lf.person_id
JOIN `{DATASET}.drug_exposure` de ON ch.person_id = de.person_id
    AND de.drug_concept_id IN (
        SELECT DISTINCT c.concept_id
        FROM `{DATASET}.concept` c
        JOIN `{DATASET}.concept_ancestor` ca ON c.concept_id = ca.descendant_concept_id
        WHERE ca.ancestor_concept_id = 10582
          AND c.domain_id = 'Drug' AND c.standard_concept = 'S'
    )
GROUP BY ch.person_id, ch.first_dx_date, lf.first_t4_date
"""

df_t4 = run_query(sql_t4_treatment)
print(f"Individuals with T4 treatment records: {len(df_t4):,}")
print(f"Fraction of hypothyroid cohort: {len(df_t4)/len(df_cohort)*100:.1f}%")
print("\nDays from first diagnosis to first T4 prescription:")
print(df_t4['days_dx_to_t4'].describe().round(1))
df_t4.head()

## 9. Treatment Response: Longitudinal TSH

TSH normalization (0.4–4.0 mIU/L) at ≥ 90 days post-treatment is the primary
biochemical definition of treatment response.

In [ ]:
sql_longitudinal_tsh = f"""
WITH {COHORT_CTE},
{LEVO_FIRST_CTE},
{TSH_CONCEPTS_CTE}
SELECT
    m.person_id,
    lf.first_t4_date,
    m.measurement_date,
    DATE_DIFF(m.measurement_date, lf.first_t4_date, DAY) AS days_from_t4_start,
    m.value_as_number AS tsh_value,
    m.range_low,
    m.range_high,
    CASE
        WHEN DATE_DIFF(m.measurement_date, lf.first_t4_date, DAY) < 0   THEN 'pre_treatment'
        WHEN DATE_DIFF(m.measurement_date, lf.first_t4_date, DAY) <= 180 THEN 'early_on_treatment'
        ELSE 'late_on_treatment'
    END AS treatment_phase
FROM `{DATASET}.measurement` m
JOIN cohort ch      ON m.person_id = ch.person_id
JOIN levo_first lf  ON m.person_id = lf.person_id
JOIN tsh_concepts tc ON m.measurement_concept_id = tc.concept_id
WHERE m.value_as_number IS NOT NULL
ORDER BY m.person_id, m.measurement_date
"""

df_tsh_long = run_query(sql_longitudinal_tsh)
print(f"Longitudinal TSH records (treated individuals): {len(df_tsh_long):,}")
print(f"Unique individuals with T4 Rx + TSH labs:       {df_tsh_long['person_id'].nunique():,}")
df_tsh_long.head()

In [ ]:
tsh_phases = df_tsh_long.groupby('person_id')['treatment_phase'].apply(set)
has_pre  = tsh_phases.apply(lambda s: 'pre_treatment' in s)
has_post = tsh_phases.apply(lambda s: 'early_on_treatment' in s or 'late_on_treatment' in s)
both = (has_pre & has_post).sum()

print(f"Individuals with pre-treatment TSH:      {has_pre.sum():,}")
print(f"Individuals with post-treatment TSH:     {has_post.sum():,}")
print(f"Individuals with BOTH pre & post TSH:    {both:,}  <-- usable for response analysis")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for phase, grp in df_tsh_long.groupby('treatment_phase'):
    ax.hist(grp['tsh_value'].clip(upper=20), bins=60, alpha=0.55, label=phase)
ax.axvline(0.4, color='green', linestyle='--', linewidth=1, label='Normal low (0.4)')
ax.axvline(4.0, color='red',   linestyle='--', linewidth=1, label='Normal high (4.0)')
ax.set_xlabel('TSH (mIU/L, clipped at 20)')
ax.set_ylabel('Count')
ax.set_title('TSH Distribution by Treatment Phase')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Treatment Response: Symptom / Condition Changes

In [ ]:
SYMPTOM_ICD10 = {
    'fatigue':               ['R53.83', 'R53.1', 'R53.81'],
    'insomnia':              ['G47.00', 'G47.09'],
    'constipation':          ['K59.00', 'K59.09'],
    'cold_intolerance':      ['R68.89'],
    'weight_gain':           ['R63.5'],
    'depression':            ['F32.9', 'F32.0', 'F32.1', 'F32.2', 'F33.0', 'F33.1', 'F33.9'],
    'dry_skin':              ['L85.3', 'R23.8'],
    'hair_loss':             ['L65.9', 'L66.9'],
    'bradycardia':           ['R00.1'],
    'cognitive_impairment':  ['R41.3', 'R41.89', 'F06.8'],
    'myalgia':               ['M79.3'],
    'edema':                 ['R60.0', 'R60.1', 'R60.9'],
    'dyslipidemia':          ['E78.5', 'E78.00', 'E78.1'],
}

mapping_rows = [
    f"('{symptom}', '{code}')"
    for symptom, codes in SYMPTOM_ICD10.items()
    for code in codes
]
mapping_values = ',\n        '.join(mapping_rows)

print(f"Symptom categories: {len(SYMPTOM_ICD10)}, ICD-10 codes: {len(mapping_rows)}")

In [ ]:
sql_symptoms = f"""
WITH {COHORT_CTE},
{LEVO_FIRST_CTE},
symptom_map AS (
    SELECT symptom_category, icd10_code
    FROM UNNEST([
        STRUCT<symptom_category STRING, icd10_code STRING>
        {mapping_values}
    ])
),
symptom_concepts AS (
    SELECT sm.symptom_category, c.concept_id
    FROM `{DATASET}.concept` c
    JOIN symptom_map sm ON c.concept_code = sm.icd10_code
    WHERE c.vocabulary_id = 'ICD10CM'
)
SELECT
    co.person_id,
    sc.symptom_category,
    co.condition_start_date,
    lf.first_t4_date,
    CASE
        WHEN lf.first_t4_date IS NULL                         THEN 'untreated'
        WHEN co.condition_start_date < lf.first_t4_date       THEN 'pre_treatment'
        ELSE 'post_treatment'
    END AS treatment_phase
FROM `{DATASET}.condition_occurrence` co
JOIN cohort ch ON co.person_id = ch.person_id
JOIN symptom_concepts sc ON co.condition_source_concept_id = sc.concept_id
LEFT JOIN levo_first lf   ON co.person_id = lf.person_id
ORDER BY co.person_id, co.condition_start_date
"""

df_symptoms = run_query(sql_symptoms)
print(f"Symptom records: {len(df_symptoms):,}")
print(f"Unique individuals with symptom data: {df_symptoms['person_id'].nunique():,}")
df_symptoms.head()

In [ ]:
symptom_pivot = (
    df_symptoms
    .groupby(['symptom_category', 'treatment_phase'])['person_id']
    .nunique()
    .unstack(fill_value=0)
    .sort_values('pre_treatment', ascending=False)
)
print("=== Unique individuals with each symptom, by treatment phase ===")
symptom_pivot

In [ ]:
if 'pre_treatment' in symptom_pivot.columns and 'post_treatment' in symptom_pivot.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(symptom_pivot))
    width = 0.35
    ax.bar([i - width/2 for i in x], symptom_pivot['pre_treatment'],  width, label='Pre-treatment')
    ax.bar([i + width/2 for i in x], symptom_pivot['post_treatment'], width, label='Post-treatment')
    ax.set_xticks(list(x))
    ax.set_xticklabels(symptom_pivot.index, rotation=40, ha='right')
    ax.set_ylabel('Unique individuals')
    ax.set_title('Hypothyroidism Symptom Prevalence: Pre vs Post T4 Treatment')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 11. Treatment Response: PPI Survey Data

In [ ]:
sql_ppi_concepts = f"""
SELECT concept_id, concept_name, concept_code, vocabulary_id
FROM `{DATASET}.concept`
WHERE vocabulary_id IN ('PPI', 'LOINC')
  AND domain_id = 'Observation'
  AND (
      LOWER(concept_name) LIKE '%fatigue%'
   OR LOWER(concept_name) LIKE '%tired%'
   OR LOWER(concept_name) LIKE '%sleep%'
   OR LOWER(concept_name) LIKE '%energy%'
   OR LOWER(concept_name) LIKE '%constipat%'
   OR LOWER(concept_name) LIKE '%cold%'
   OR LOWER(concept_name) LIKE '%concentrate%'
   OR LOWER(concept_name) LIKE '%memory%'
   OR LOWER(concept_name) LIKE '%weight%'
  )
ORDER BY vocabulary_id, concept_name
"""

df_ppi_concepts = run_query(sql_ppi_concepts)
print(f"PPI/survey observation concepts found: {len(df_ppi_concepts)}")
df_ppi_concepts

In [ ]:
if len(df_ppi_concepts) > 0:
    ppi_concept_ids = ', '.join(str(i) for i in df_ppi_concepts['concept_id'].tolist())

    sql_ppi_obs = f"""
    WITH {COHORT_CTE}
    SELECT
        o.person_id,
        o.observation_date,
        c.concept_name  AS symptom_question,
        c.vocabulary_id,
        vc.concept_name AS answer,
        o.value_as_number,
        o.value_as_string
    FROM `{DATASET}.observation` o
    JOIN cohort ch ON o.person_id = ch.person_id
    JOIN `{DATASET}.concept` c ON o.observation_concept_id = c.concept_id
    LEFT JOIN `{DATASET}.concept` vc ON o.value_as_concept_id = vc.concept_id
    WHERE o.observation_concept_id IN ({ppi_concept_ids})
    ORDER BY o.person_id, o.observation_date
    """

    df_ppi_obs = run_query(sql_ppi_obs)
    print(f"PPI observation records for cohort: {len(df_ppi_obs):,}")
    print(f"Individuals with PPI survey data:   {df_ppi_obs['person_id'].nunique():,}")
    df_ppi_obs.head()
else:
    df_ppi_obs = pd.DataFrame()
    print("No PPI/survey concepts found — check vocabulary availability in this CDR version.")

## 12. Genomic & Multi-Omic Data Availability

In [ ]:
sql_data_avail = f"""
WITH {COHORT_CTE}
SELECT
    COUNTIF(sp.has_ehr_data = 1)             AS n_ehr,
    COUNTIF(sp.has_physical_measurement = 1) AS n_physical_measurement,
    COUNTIF(sp.has_whole_genome_variant = 1) AS n_wgs,
    COUNTIF(sp.has_array_data = 1)           AS n_genotyping_array,
    COUNTIF(sp.has_survey_data = 1)          AS n_survey,
    COUNTIF(sp.has_fitbit_data = 1)          AS n_fitbit,
    COUNT(*)                                  AS n_total_cohort
FROM `{DATASET}.cb_search_person` sp
JOIN cohort ch ON sp.person_id = ch.person_id
"""

try:
    df_avail = run_query(sql_data_avail)
    print("=== Data Modality Availability for Hypothyroid Cohort ===")
    display(df_avail.T.rename(columns={0: 'count'}))
except Exception as e:
    print(f"cb_search_person query error: {e}")

## 13. Non-Responder Classification

Classify treated individuals using post-treatment TSH (≥ 90 days after first T4 prescription):
- **responder** — all post-Rx TSH values within normal range (0.4–4.0)
- **non_responder** — all post-Rx TSH values elevated (> 4.0)
- **partial_responder** — mixed normal and elevated post-Rx TSH values
- **insufficient_data** — fewer than 2 post-Rx TSH values

Four additional non-responder flags are stored as boolean columns in `df_response`:

- **t3_nonresponder** — person had ≥ 1 TSH > 4.0 and was subsequently prescribed liothyronine (T3).

- **count_symptom_nonresponder** — for at least one of fatigue, cognitive impairment, or cold
  intolerance, the number of ICD-10 diagnosis code occurrences in `condition_occurrence` did not
  decrease after T4 start (post-count ≥ pre-count, with pre-count > 0).

- **ppi_score_nonresponder** — for at least one target symptom domain, the mean numeric PPI survey
  score (`observation.value_as_number`) did not decrease after T4 start. Requires numeric survey
  responses both before and after treatment.

- **ppi_unreported_nonresponder** — for at least one target symptom domain, the person had a PPI
  survey response before T4 but no survey response for that domain after T4, meaning improvement
  could not be confirmed from self-reported data.

In [ ]:
sql_nonresponders = f"""
WITH {COHORT_CTE},
{LEVO_FIRST_CTE},
{TSH_CONCEPTS_CTE},
{T3_FIRST_CTE},
all_tsh AS (
    -- All TSH values for cohort members (used for T3 non-responder check)
    SELECT
        m.person_id,
        m.measurement_date,
        m.value_as_number AS tsh_value
    FROM `{DATASET}.measurement` m
    JOIN cohort ch ON m.person_id = ch.person_id
    JOIN tsh_concepts tc ON m.measurement_concept_id = tc.concept_id
    WHERE m.value_as_number IS NOT NULL
),
post_tsh AS (
    -- TSH values >= 90 days after first T4 prescription
    SELECT
        m.person_id,
        m.value_as_number AS tsh_value
    FROM `{DATASET}.measurement` m
    JOIN cohort ch     ON m.person_id = ch.person_id
    JOIN levo_first lf ON m.person_id = lf.person_id
    JOIN tsh_concepts tc ON m.measurement_concept_id = tc.concept_id
    WHERE m.value_as_number IS NOT NULL
      AND DATE_DIFF(m.measurement_date, lf.first_t4_date, DAY) >= 90
),
t3_after_high_tsh AS (
    -- Persons who had at least one high TSH (> 4.0) and were then prescribed T3
    SELECT DISTINCT at.person_id
    FROM all_tsh at
    JOIN t3_first tf ON at.person_id = tf.person_id
    WHERE at.tsh_value > 4.0
      AND at.measurement_date < tf.first_t3_date
),
per_person AS (
    SELECT
        person_id,
        COUNT(*)              AS n_post_tsh,
        COUNTIF(tsh_value > 4.0) AS n_elevated,
        AVG(tsh_value)        AS mean_post_tsh,
        MIN(tsh_value)        AS min_post_tsh,
        MAX(tsh_value)        AS max_post_tsh
    FROM post_tsh
    GROUP BY person_id
)
SELECT
    pp.*,
    CASE
        WHEN pp.n_post_tsh >= 2 AND pp.n_elevated = pp.n_post_tsh THEN 'non_responder'
        WHEN pp.n_post_tsh >= 2 AND pp.n_elevated = 0             THEN 'responder'
        WHEN pp.n_post_tsh >= 2                                    THEN 'partial_responder'
        ELSE 'insufficient_data'
    END AS response_category,
    -- T3 non-responder: high TSH followed by T3 prescription (independent of T4 response class)
    CASE WHEN taht.person_id IS NOT NULL THEN TRUE ELSE FALSE END AS t3_nonresponder
FROM per_person pp
LEFT JOIN t3_after_high_tsh taht ON pp.person_id = taht.person_id
ORDER BY pp.mean_post_tsh DESC
"""

df_response = run_query(sql_nonresponders)
print("=== Treatment Response Classification ===")
print(df_response['response_category'].value_counts())
print(f"\nT3 non-responders (high TSH → T3 prescribed): {df_response['t3_nonresponder'].sum():,}")
print("\nT3 non-responder flag cross-tabulated with TSH-based category:")
print(df_response.groupby(['response_category', 't3_nonresponder']).size().unstack(fill_value=0))
df_response.head(20)

In [ ]:
# ---------------------------------------------------------------------------
# Non-responder class 2: Symptom count-based
# For each of the three target symptoms, count condition_occurrence records
# before and after first T4 date. A person is flagged if post-count >= pre-count
# for at least one symptom (occurrences did not decrease after treatment).
# ---------------------------------------------------------------------------
TARGET_SYMPTOMS = ['fatigue', 'cognitive_impairment', 'cold_intolerance']
target_rows = [
    f"('{symptom}', '{code}')"
    for symptom in TARGET_SYMPTOMS
    for code in SYMPTOM_ICD10[symptom]
]
target_mapping_values = ',\n        '.join(target_rows)

sql_count_nonresponders = f"""
WITH {COHORT_CTE},
{LEVO_FIRST_CTE},
target_symptom_map AS (
    SELECT symptom_category, icd10_code
    FROM UNNEST([
        STRUCT<symptom_category STRING, icd10_code STRING>
        {target_mapping_values}
    ])
),
target_symptom_concepts AS (
    SELECT tsm.symptom_category, c.concept_id
    FROM `{DATASET}.concept` c
    JOIN target_symptom_map tsm ON c.concept_code = tsm.icd10_code
    WHERE c.vocabulary_id = 'ICD10CM'
),
symptom_counts AS (
    SELECT
        co.person_id,
        tsc.symptom_category,
        COUNTIF(co.condition_start_date <  lf.first_t4_date) AS pre_count,
        COUNTIF(co.condition_start_date >= lf.first_t4_date) AS post_count
    FROM `{DATASET}.condition_occurrence` co
    JOIN cohort ch ON co.person_id = ch.person_id
    JOIN levo_first lf ON co.person_id = lf.person_id
    JOIN target_symptom_concepts tsc ON co.condition_source_concept_id = tsc.concept_id
    GROUP BY co.person_id, tsc.symptom_category
),
count_nonresponders AS (
    -- At least one symptom had pre-treatment occurrences and count did not fall
    SELECT DISTINCT person_id
    FROM symptom_counts
    WHERE pre_count > 0
      AND post_count >= pre_count
)
SELECT person_id, TRUE AS count_symptom_nonresponder
FROM count_nonresponders
"""

df_count_nr = run_query(sql_count_nonresponders)
print(f"Count-based symptom non-responders: {len(df_count_nr):,}")

df_response = df_response.merge(df_count_nr, on='person_id', how='left')
df_response['count_symptom_nonresponder'] = df_response['count_symptom_nonresponder'].fillna(False)

print("\nCount symptom non-responder × TSH response category:")
print(df_response.groupby(['response_category', 'count_symptom_nonresponder']).size().unstack(fill_value=0))

In [ ]:
# ---------------------------------------------------------------------------
# Non-responder class 3: PPI survey score-based
# For each of the three symptom domains, compare mean numeric PPI survey score
# (observation.value_as_number) before and after first T4 date. A person is
# flagged if mean post-score >= mean pre-score for at least one domain, meaning
# the self-reported symptom burden did not decrease after treatment.
#
# PPI concepts are matched by keyword within the domain:
#   fatigue       — 'fatigue', 'tired', 'energy'
#   cognitive     — 'concentrate', 'memory', 'cognitive'
#   cold          — 'cold'
# ---------------------------------------------------------------------------
sql_ppi_nonresponders = f"""
WITH {COHORT_CTE},
{LEVO_FIRST_CTE},
ppi_target_concepts AS (
    SELECT
        concept_id,
        CASE
            WHEN LOWER(concept_name) LIKE '%fatigue%'
              OR LOWER(concept_name) LIKE '%tired%'
              OR LOWER(concept_name) LIKE '%energy%'   THEN 'fatigue'
            WHEN LOWER(concept_name) LIKE '%concentrate%'
              OR LOWER(concept_name) LIKE '%memory%'
              OR LOWER(concept_name) LIKE '%cognitive%' THEN 'cognitive_impairment'
            WHEN LOWER(concept_name) LIKE '%cold%'      THEN 'cold_intolerance'
        END AS symptom_domain
    FROM `{DATASET}.concept`
    WHERE vocabulary_id IN ('PPI', 'LOINC')
      AND domain_id = 'Observation'
      AND (
          LOWER(concept_name) LIKE '%fatigue%'
       OR LOWER(concept_name) LIKE '%tired%'
       OR LOWER(concept_name) LIKE '%energy%'
       OR LOWER(concept_name) LIKE '%concentrate%'
       OR LOWER(concept_name) LIKE '%memory%'
       OR LOWER(concept_name) LIKE '%cognitive%'
       OR LOWER(concept_name) LIKE '%cold%'
      )
),
ppi_scores AS (
    SELECT
        o.person_id,
        ptc.symptom_domain,
        AVG(CASE WHEN o.observation_date <  lf.first_t4_date THEN o.value_as_number END) AS mean_pre_score,
        AVG(CASE WHEN o.observation_date >= lf.first_t4_date THEN o.value_as_number END) AS mean_post_score
    FROM `{DATASET}.observation` o
    JOIN cohort ch ON o.person_id = ch.person_id
    JOIN levo_first lf ON o.person_id = lf.person_id
    JOIN ppi_target_concepts ptc ON o.observation_concept_id = ptc.concept_id
    WHERE o.value_as_number IS NOT NULL
      AND ptc.symptom_domain IS NOT NULL
    GROUP BY o.person_id, ptc.symptom_domain
),
ppi_nonresponders AS (
    -- At least one domain: had both pre and post scores, and post mean >= pre mean
    SELECT DISTINCT person_id
    FROM ppi_scores
    WHERE mean_pre_score IS NOT NULL
      AND mean_post_score IS NOT NULL
      AND mean_post_score >= mean_pre_score
)
SELECT person_id, TRUE AS ppi_symptom_nonresponder
FROM ppi_nonresponders
"""

df_ppi_nr = run_query(sql_ppi_nonresponders)
print(f"PPI score-based symptom non-responders: {len(df_ppi_nr):,}")

df_response = df_response.merge(df_ppi_nr, on='person_id', how='left')
df_response['ppi_symptom_nonresponder'] = df_response['ppi_symptom_nonresponder'].fillna(False)

print("\nPPI symptom non-responder × TSH response category:")
print(df_response.groupby(['response_category', 'ppi_symptom_nonresponder']).size().unstack(fill_value=0))

In [ ]:
# ---------------------------------------------------------------------------
# Non-responder class 4: PPI survey presence-based (unreported post-treatment)
# A person is flagged if, for at least one target symptom domain, they had a
# PPI survey response BEFORE T4 treatment but NO survey response for that
# same domain AFTER T4 treatment — meaning improvement was never reported.
# ---------------------------------------------------------------------------
sql_ppi_unreported_nonresponders = f"""
WITH {COHORT_CTE},
{LEVO_FIRST_CTE},
ppi_target_concepts AS (
    SELECT
        concept_id,
        CASE
            WHEN LOWER(concept_name) LIKE '%fatigue%'
              OR LOWER(concept_name) LIKE '%tired%'
              OR LOWER(concept_name) LIKE '%energy%'   THEN 'fatigue'
            WHEN LOWER(concept_name) LIKE '%concentrate%'
              OR LOWER(concept_name) LIKE '%memory%'
              OR LOWER(concept_name) LIKE '%cognitive%' THEN 'cognitive_impairment'
            WHEN LOWER(concept_name) LIKE '%cold%'      THEN 'cold_intolerance'
        END AS symptom_domain
    FROM `{DATASET}.concept`
    WHERE vocabulary_id IN ('PPI', 'LOINC')
      AND domain_id = 'Observation'
      AND (
          LOWER(concept_name) LIKE '%fatigue%'
       OR LOWER(concept_name) LIKE '%tired%'
       OR LOWER(concept_name) LIKE '%energy%'
       OR LOWER(concept_name) LIKE '%concentrate%'
       OR LOWER(concept_name) LIKE '%memory%'
       OR LOWER(concept_name) LIKE '%cognitive%'
       OR LOWER(concept_name) LIKE '%cold%'
      )
),
survey_presence AS (
    -- For each person × domain, record whether any response exists pre and post T4
    SELECT
        o.person_id,
        ptc.symptom_domain,
        COUNTIF(o.observation_date <  lf.first_t4_date) AS pre_response_count,
        COUNTIF(o.observation_date >= lf.first_t4_date) AS post_response_count
    FROM `{DATASET}.observation` o
    JOIN cohort ch ON o.person_id = ch.person_id
    JOIN levo_first lf ON o.person_id = lf.person_id
    JOIN ppi_target_concepts ptc ON o.observation_concept_id = ptc.concept_id
    WHERE ptc.symptom_domain IS NOT NULL
    GROUP BY o.person_id, ptc.symptom_domain
),
ppi_unreported_nonresponders AS (
    -- Symptom reported pre-treatment but no survey response recorded post-treatment
    SELECT DISTINCT person_id
    FROM survey_presence
    WHERE pre_response_count > 0
      AND post_response_count = 0
)
SELECT person_id, TRUE AS ppi_unreported_nonresponder
FROM ppi_unreported_nonresponders
"""

df_ppi_unreported_nr = run_query(sql_ppi_unreported_nonresponders)
print(f"PPI unreported non-responders (symptom reported pre, no post survey): {len(df_ppi_unreported_nr):,}")

df_response = df_response.merge(df_ppi_unreported_nr, on='person_id', how='left')
df_response['ppi_unreported_nonresponder'] = df_response['ppi_unreported_nonresponder'].fillna(False)

print("\nPPI unreported non-responder × TSH response category:")
print(df_response.groupby(['response_category', 'ppi_unreported_nonresponder']).size().unstack(fill_value=0))

## 14. Data Availability Summary

In [ ]:
print("====== DATA AVAILABILITY SUMMARY ======")
print(f"1. Hypothyroid cohort (total):           {len(df_cohort):>10,}")
print(f"2. With >= 1 thyroid lab result:         {df_labs['person_id'].nunique():>10,}")
print(f"3. With T4 (levothyroxine) treatment:    {len(df_t4):>10,}")
print(f"4. With pre + post TSH (response eval):  {both:>10,}")
print(f"5. With symptom code data:               {df_symptoms['person_id'].nunique():>10,}")
if not df_ppi_obs.empty:
    print(f"6. With PPI/survey symptom data:         {df_ppi_obs['person_id'].nunique():>10,}")
print()
print("Response classification breakdown:")
for cat, cnt in df_response['response_category'].value_counts().items():
    print(f"   {cat:<25} {cnt:,}")